# core

> User facing API for solvedrive

In [ ]:
#| default_exp core

## Imports

In [ ]:
from fastcore.all import *
from fastgws import Drive as GDrive
from fastgws.auth import *
from uuid import uuid4

import json, mimetypes, sys

In [ ]:
from fastcore.test import *
import asyncio

Google Drive's API is a single REST API organized around one central resource type: the **file**. Everything in Drive is a file: documents, images, even folders (a folder is a file with a special `mimeType`). Files are searched with Drive's own [query syntax](https://developers.google.com/workspace/drive/api/guides/search-files) (`name contains 'report'`, `mimeType='application/pdf'`, `'<folder-id>' in parents`), and the server returns **partial resources**. Unless you ask for specific `fields`, you get only `id`, `name`, `mimeType` and `kind` back.

A file's *content* comes in two flavors: regular binary blobs (downloaded as-is), and Google-native files (Docs, Sheets, Slides) which have no bytes of their own and must be *exported* to a regular format.

In [ ]:
FIELDS = 'id,name,mimeType,size,modifiedTime,parents,trashed,webViewLink'

class Drive:
    def __init__(self, client): self.client = client

    @classmethod
    async def init(cls, scopes='readonly', creds=None, redirect_uri='https://oauth.appapis.org/redirect',
        listen=False, port=0, open_url=print):
        if scopes == 'full': scopes = ['https://www.googleapis.com/auth/drive']
        else:                scopes = [f'https://www.googleapis.com/auth/drive.{x}' for x in listify(scopes)]
        if creds is None: creds = await oauth_creds(scopes=scopes, redirect_uri=redirect_uri, listen=listen, port=port, open_url=open_url)
        return cls(GDrive(creds=creds))

    async def about(self):
        "User resource with `email` attribute"
        d = await self.client.about.get(fields='user')
        return AttrDict(d.user, email=d.user.emailAddress)

In [ ]:
drive = await Drive.init(scopes='full')
about = await drive.about()
me = about.email
me

'nc@answer.ai'

### Files

A file is Drive's central noun. `File` wraps the API's file resource, fetching our default `FIELDS` so the useful attributes (`name`, `mimeType`, `size`, `parents`, `trashed`, `webViewLink`) are always on hand. Anything else can be requested via `fields=`. `Folder` is a `File` whose `mimeType` marks it as a folder, and `_mk_file` picks the right class.

In [ ]:
FOLDER_MIME = 'application/vnd.google-apps.folder'

def hsize(n):
    "Human-readable byte size"
    for u in ' KMGT':
        if n < 1024: return f'{n:.0f}{u.strip()}B' if u==' ' else f'{n:.1f}{u}B'
        n /= 1024
    return f'{n:.1f}PB'

def _mtype(m): return m.split('.')[-1] if m.startswith('application/vnd.google-apps') else m

def _link(f):
    "File name as an anchor opening it in Drive in a new tab"
    nm = f.get('name') or ''
    return f'<a href="{url}" target="_blank">{nm}</a>' if (url:=f.get('webViewLink')) else nm

class File(AttrDict):
    def __init__(self, drive, d):
        super().__init__(d)
        self.drive = drive

    def _desc(self, nm):
        sz = f', {hsize(int(self.size))}' if self.get('size') else ''
        trash = '🗑' if self.get('trashed') else ''
        return f'{type(self).__name__}({self.id}: {trash}{nm} ({_mtype(self.get("mimeType", ""))}{sz}) {self.get("modifiedTime", "")})'

    def __repr__(self): return self._desc(self.get('name', ''))
    def _repr_markdown_(self): return self._desc(_link(self))

    async def refresh(self, fields=FIELDS):
        self.update(await self.drive.client.files.get(file_id=self.id, fields=fields))
        return self

    @classmethod
    async def fetch(cls, drive, id): return await cls(drive, dict(id=id)).refresh()

class Folder(File): pass

def _fid(f): return f.id if isinstance(f, File) else f

def _mk_file(drive, d):
    cls = Folder if d.get('mimeType')==FOLDER_MIME else File
    return cls(drive, d)

### Uploading

Uploads use a separate endpoint (`/upload/drive/v3/files`) with a `multipart/related` body carrying the JSON metadata and the file content in one request. That protocol caps multipart uploads at 5MB; bigger files need Google's resumable protocol, which we don't support (yet).

In [ ]:
_upload_url = 'https://www.googleapis.com/upload/drive/v3/files'

def _multipart_related(meta, data, mime):
    b = uuid4().hex
    body = (f'--{b}\r\nContent-Type: application/json; charset=UTF-8\r\n\r\n{json.dumps(meta)}\r\n'
        f'--{b}\r\nContent-Type: {mime}\r\n\r\n').encode() + data + f'\r\n--{b}--'.encode()
    return body, {'Content-Type': f'multipart/related; boundary={b}'}

@patch
async def upload(self:Drive,
    path=None,   # File to upload
    name=None,   # Name in Drive (defaults to `path`'s name)
    data=None,   # Content bytes, as an alternative to `path`
    mime=None,   # Content type (guessed from `name` if not given)
    folder=None, # Destination `Folder` or folder id (default: My Drive root)
):
    "Upload a file to Drive, returning the new `File`"
    if path:
        p = Path(path)
        name, data = name or p.name, p.read_bytes()
    mime = mime or mimetypes.guess_type(name)[0] or 'application/octet-stream'
    meta = dict(name=name)
    if folder: meta['parents'] = [_fid(folder)]
    body, hdrs = _multipart_related(meta, data, mime)
    r = await self.client.transport.request('POST', _upload_url, content=body, headers=hdrs,
        params=dict(uploadType='multipart', fields=FIELDS))
    return _mk_file(self, r)

In [ ]:
f = await drive.upload(data=b'hello from solvedrive!', name='solvedrive_test.txt')
test_eq(f.name, 'solvedrive_test.txt')
f

File(1fYKN13AVZaNGZ3UzfTpIv1P6marr7mso: <a href="https://drive.google.com/file/d/1fYKN13AVZaNGZ3UzfTpIv1P6marr7mso/view?usp=drivesdk" target="_blank">solvedrive_test.txt</a> (text/plain, 22B) 2026-07-28T13:34:43.131Z)

`Files` is a collection of `File`s with a table repr, and `search_files` runs a Drive query, returning results with our default `FIELDS` already populated (one request, no follow-up fetches needed).

In [ ]:
def _file_row(f):
    sz = hsize(int(f.size)) if f.get('size') else ''
    return f'| {f.id} | {"🗑" if f.get("trashed") else ""}{_link(f)} | {_mtype(f.get("mimeType") or "")} | {sz} | {(f.get("modifiedTime") or "")[:10]} |'

_file_hdr = '| id | name | type | size | modified |\n|-|-|-|-|-|\n'
class Files(L):
    def __init__(self, d, *rest, drive=None, **kwargs):
        super().__init__(d, *rest, **kwargs)
        self.drive = drive

    def _repr_markdown_(self): return _file_hdr + '\n'.join(self.map(_file_row))

@patch
async def search_files(self:Drive, q=None, max_results=20, fields=FIELDS, **kwargs):
    r = await self.client.files.list(q=q, page_size=max_results, fields=f'files({fields})', **kwargs)
    return Files(L(r.get('files') or []).map(lambda o: _mk_file(self, o)), drive=self)

Search takes Drive's [query syntax](https://developers.google.com/workspace/drive/api/guides/search-files) directly: `name contains 'report'`, `fullText contains 'quarterly'`, `mimeType='application/pdf'`, `modifiedTime > '2026-01-01'`, `starred=true`, combined with `and`/`or`/`not`. Trashed files match too unless you exclude them with `trashed=false`.

In [ ]:
fs = await drive.search_files("name = 'solvedrive_test.txt' and trashed=false")
fs

| id | name | type | size | modified |
|-|-|-|-|-|
| 1fYKN13AVZaNGZ3UzfTpIv1P6marr7mso | <a href="https://drive.google.com/file/d/1fYKN13AVZaNGZ3UzfTpIv1P6marr7mso/view?usp=drivesdk" target="_blank">solvedrive_test.txt</a> | text/plain | 22B | 2026-07-28 |

### Content

`download` returns a file's bytes, dispatching on the file's type so you don't have to: regular files download as-is, while Google-native files (which have no bytes of their own) are *exported*. A Doc becomes a docx, a Sheet an xlsx, a Slides deck a pptx. Pass `mime` to export to a different format (Google's [export table](https://developers.google.com/workspace/drive/api/guides/ref-export-formats) lists what's supported); it's an error on a regular file, which isn't converted. Note exports are capped at 10MB by the API.

In [ ]:
_exports = {'application/vnd.google-apps.document': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document',
    'application/vnd.google-apps.spreadsheet': 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet',
    'application/vnd.google-apps.presentation': 'application/vnd.openxmlformats-officedocument.presentationml.presentation',
    'application/vnd.google-apps.drawing': 'image/png'}

@patch
async def download(self:File,
    save=None, # Optional path to also write the bytes to (a directory uses the file's name)
    mime=None, # Export format for Google-native files
):
    "File content as bytes; Google-native files are exported (docx/xlsx/pptx by default)"
    native = self.mimeType.startswith('application/vnd.google-apps')
    if not native:
        if mime: raise ValueError(f'`mime` is for exporting Google-native files; {self.name} is {self.mimeType}')
        r = await self.drive.client.files.get(file_id=self.id, alt='media', raw=True)
    else: r = await self.drive.client.files.export(file_id=self.id, mime_type=mime or _exports[self.mimeType], raw=True)
    data = r.content
    if save:
        p = Path(save)
        if p.is_dir(): p = p/self.name
        p.write_bytes(data)
    return data

In [ ]:
test_eq(await f.download(), b'hello from solvedrive!')

Google-native files can be created through `files.create` with the native `mimeType` (no upload needed, since there are no bytes). Exporting one gives a regular file in the requested format: docx and friends are zip containers, so the bytes start with `PK`.

In [ ]:
doc = _mk_file(drive, await drive.client.files.create(name='solvedrive test doc',
    mime_type='application/vnd.google-apps.document', fields=FIELDS))
docx = await doc.download()
test_eq(docx[:2], b'PK')
doc

File(1xbe4E2kGLrA9E8YCBWBUvma7WP2-zOvh9eSfVJn4IJQ: <a href="https://docs.google.com/document/d/1xbe4E2kGLrA9E8YCBWBUvma7WP2-zOvh9eSfVJn4IJQ/edit?usp=drivesdk" target="_blank">solvedrive test doc</a> (document, 1B) 2026-07-28T13:35:08.969Z)

### Folders

A folder is a file with the folder `mimeType`. It has no content, and "containment" is the child listing the folder in its `parents`. So `Folder` needs no new state: `ls` is a search scoped to `'<id>' in parents`, and `upload` drops files straight into it.

In [ ]:
@patch
async def create_folder(self:Drive, name, parent=None):
    "Create a folder, returning the new `Folder`"
    meta = dict(name=name, mime_type=FOLDER_MIME, fields=FIELDS)
    if parent: meta['parents'] = [_fid(parent)]
    return Folder(self, await self.client.files.create(**meta))

@patch
async def ls(self:Folder, q=None, **kwargs):
    "Search within this folder"
    qs = f"'{self.id}' in parents and trashed=false"
    if q: qs = f'{qs} and ({q})'
    return await self.drive.search_files(qs, **kwargs)

@patch
@delegates(Drive.upload)
async def upload(self:Folder, path=None, **kwargs):
    "Upload into this folder"
    return await self.drive.upload(path, folder=self, **kwargs)

In [ ]:
folder = await drive.create_folder('solvedrive tests')
f2 = await folder.upload(data=b'nested file', name='nested.txt')
kids = await folder.ls()
test_eq(kids.attrgot('id'), [f2.id])
kids

| id | name | type | size | modified |
|-|-|-|-|-|
| 1SZ1IbceAFoQKo_MS2iTfANwV0Q85dBdk | <a href="https://drive.google.com/file/d/1SZ1IbceAFoQKo_MS2iTfANwV0Q85dBdk/view?usp=drivesdk" target="_blank">nested.txt</a> | text/plain | 11B | 2026-07-28 |

### Organizing

Renaming is a metadata update; moving updates `parents` (`add_parents`/`remove_parents` in one call); copying is server-side, so no bytes travel through your machine. Each returns the updated (or new) `File`.

In [ ]:
@patch
async def rename(self:File, name):
    "Rename to `name`"
    return self.update(await self.drive.client.files.update(file_id=self.id, name=name, fields=FIELDS))

@patch
async def move(self:File, folder):
    "Move into `folder` (a `Folder` or folder id)"
    return self.update(await self.drive.client.files.update(file_id=self.id, add_parents=_fid(folder),
        remove_parents=','.join(self.get('parents') or []), fields=FIELDS))

@patch
async def copy(self:File, name=None, folder=None):
    "Server-side copy, optionally renamed into `folder`; returns the new `File`"
    meta = dict(file_id=self.id, fields=FIELDS)
    if name: meta['name'] = name
    if folder: meta['parents'] = [_fid(folder)]
    return _mk_file(self.drive, await self.drive.client.files.copy(**meta))

In [ ]:
await f.rename('solvedrive_renamed.txt')
await f.move(folder)
test_eq(f.name, 'solvedrive_renamed.txt')
test_eq(f.parents, [folder.id])

In [ ]:
c = await f.copy(name='solvedrive_copy.txt', folder=folder)
test_eq(await c.download(), await f.download())
c

File(1xgvir1NLj-_UbblS78VvAaU6qXc23Q01: <a href="https://drive.google.com/file/d/1xgvir1NLj-_UbblS78VvAaU6qXc23Q01/view?usp=drivesdk" target="_blank">solvedrive_copy.txt</a> (text/plain, 22B) 2026-07-28T13:35:52.291Z)

### Trash and deleting

Drive treats trashing and deletion differently. Trashing flips the `trashed` flag: the file stays (and still matches searches unless you filter `trashed=false`) until Drive purges it after 30 days, and `untrash` restores it. `delete` removes the file permanently and immediately, with no trash and no undo. Deleting a folder deletes everything inside it.

In [ ]:
@patch
async def _set_trashed(self:File, trashed):
    return self.update(await self.drive.client.files.update(file_id=self.id, trashed=trashed, fields=FIELDS))

@patch
async def trash(self:File):
    "Move to the trash (recoverable for 30 days)"
    sys.audit('solvedrive.File.trash')
    return await self._set_trashed(True)

@patch
async def untrash(self:File):
    "Restore from the trash"
    return await self._set_trashed(False)

@patch
async def delete(self:File):
    "Delete permanently, skipping the trash"
    sys.audit('solvedrive.File.delete')
    return await self.drive.client.files.delete(file_id=self.id)

In [ ]:
await c.trash()
assert c.trashed
await c.untrash()
assert not c.trashed
await c.delete()
test_eq(await folder.ls('name = "solvedrive_copy.txt"'), [])

### Batch operations

Operating on many files one request at a time is slow. Google's batch protocol packs up to 100 requests into one HTTP round trip, and `Files` uses it for `refresh`, `trash`, `untrash`, and `delete`, so a search result can be acted on as a unit.

In [ ]:
@patch
def _gfiles(self:Files): return self.drive.client._gservice.files()

@patch
async def refresh(self:Files, fields=FIELDS):
    "Batch re-fetch metadata for all files"
    if not self: return self
    res = self.drive.client.batch_get([(f.id, self._gfiles().get(fileId=f.id, fields=fields)) for f in self])
    for f in self: f.update(dict2obj(res[f.id]))
    return self

@patch
async def _set_trashed(self:Files, trashed):
    if not self: return self
    res = self.drive.client.batch_get([(f.id, self._gfiles().update(fileId=f.id, body=dict(trashed=trashed), fields=FIELDS)) for f in self])
    for f in self: f.update(dict2obj(res[f.id]))
    return self

@patch
async def trash(self:Files):
    "Batch move all files to the trash"
    sys.audit('solvedrive.Files.trash')
    return await self._set_trashed(True)

@patch
async def untrash(self:Files):
    "Batch restore all files from the trash"
    return await self._set_trashed(False)

@patch
async def delete(self:Files):
    "Batch delete all files permanently"
    sys.audit('solvedrive.Files.delete')
    if self: self.drive.client.batch_get([(f.id, self._gfiles().delete(fileId=f.id)) for f in self])

In [ ]:
ups = await asyncio.gather(*[folder.upload(data=f'test {i}'.encode(), name=f'batch_{i}.txt') for i in range(3)])
fs = Files(ups, drive=drive)
await fs.trash()
test_eq(fs.attrgot('trashed'), [True]*3)
await fs.untrash()
test_eq(fs.attrgot('trashed'), [False]*3)
await fs.delete()
test_eq(await folder.ls("name contains 'batch_'"), [])

### Sharing

Access is a list of **permissions** on a file: each grants a `role` (`reader`, `commenter`, `writer`, or `owner`) to a grantee `type` (`user`, `group`, `domain`, or `anyone`). `share` grants to an email, or to anyone-with-the-link when no email is given, after which `webViewLink` is shareable.

In [ ]:
@patch
async def share(self:File,
    email=None,      # User or group to share with (anyone-with-the-link if None)
    role='reader',   # 'reader', 'commenter', 'writer', or 'owner'
    notify=False,    # Send the grantee a notification email?
):
    "Grant access, returning the new permission"
    sys.audit('solvedrive.File.share')
    meta = dict(file_id=self.id, role=role, send_notification_email=notify)
    if email: meta.update(type='user', email_address=email)
    else: meta['type'] = 'anyone'
    return await self.drive.client.permissions.create(**meta)

@patch(as_prop=True)
async def permissions(self:File):
    r = await self.drive.client.permissions.list(file_id=self.id, fields='permissions(id,type,role,emailAddress)')
    return r.permissions

@patch
async def unshare(self:File, permission_id):
    "Revoke a permission by id"
    return await self.drive.client.permissions.delete(file_id=self.id, permission_id=permission_id)

In [ ]:
p = await f.share()
test_eq(p.type, 'anyone')
test_eq(p.role, 'reader')
await f.permissions

[{'id': 'anyoneWithLink', 'type': 'anyone', 'role': 'reader'}, {'id': '12635610242443046038', 'type': 'user', 'emailAddress': 'nc@answer.ai', 'role': 'owner'}]

In [ ]:
await f.unshare(p.id)
test_eq(len(await f.permissions), 1)

Finally, clean up everything this notebook created (deleting the folder deletes its contents too):

In [ ]:
await folder.delete()
await doc.delete()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()